# E-ComShield — inspeção e limpeza inicial do TalkEx

Este notebook documenta a organização, inspeção e limpeza mínima do dataset usado no TP1. O arquivo original permanece imutável em `data/raw/`; somente o resultado de uma transformação justificada é salvo em `data/processed/`.

## Documentação do dataset

| Item | Informação |
|---|---|
| Nome | TalkEx Augmented PT-BR |
| Fonte | Hugging Face |
| URL | https://huggingface.co/datasets/paulohenriquevn/talkex-augmented-pt-br |
| Licença | Apache 2.0 |
| Idioma | Português brasileiro |

A escolha é adequada ao E-ComShield porque reúne conversas de atendimento em PT-BR, possui texto (`text`) e intenção (`intent`), e inclui o setor `ecommerce`. As limitações são a presença de dados sintéticos, o número relativamente pequeno de intenções (8) e a possível ausência de intenções futuras específicas do E-ComShield.

## 1. Imports e caminhos

Os caminhos são relativos à raiz identificada pelo `pyproject.toml`; não há dependência de caminhos da máquina local.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('pyproject.toml não foi encontrado.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'talkex_augmented_pt_br.parquet'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'talkex_clean.csv'

print(f'Dataset bruto: {RAW_DATA_PATH.relative_to(PROJECT_ROOT)}')

Dataset bruto: data/raw/talkex_augmented_pt_br.parquet


## 2. Carregamento do arquivo original

O arquivo oficial é disponibilizado em formato Parquet. Ele é apenas lido nesta etapa.

In [2]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {RAW_DATA_PATH}')

df = pd.read_parquet(RAW_DATA_PATH)
print(f'Arquivo: {RAW_DATA_PATH.name}')
print(f'Tamanho: {RAW_DATA_PATH.stat().st_size / 1024:.1f} KiB')
print(f'Registros carregados: {len(df):,}')

Arquivo: talkex_augmented_pt_br.parquet
Tamanho: 924.1 KiB
Registros carregados: 2,120


## 3. Inspeção estrutural

Esta seção apresenta exemplos, dimensões, colunas, tipos e resumo descritivo — evidências centrais para a rubrica.

In [3]:
display(df.head())
display(df.sample(n=5, random_state=42))
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
print('\nInformações do DataFrame:')
df.info()
print('\nTipos de dados:')
display(df.dtypes.rename('dtype').to_frame())
print('\nResumo descritivo:')
display(df.describe(include='all').T)

,conv_id,text,intent,sector,sentiment,turns,_synthetic
0,conv_00384,"[customer] Oi, vc pode me ajudar? Quero contra...",compra,telecom,negative,8,False
1,conv_00893,"[customer] Oi, tudo bem? vc pode me ajudar com...",reclamacao,tecnologia,positive,8,False
2,conv_00398,"[customer] Olá, vc pode me ajudar a cancelar o...",cancelamento,telecom,negative,8,False
3,conv_00652,"[customer] oi, vc tem aquele lanche de frango ...",duvida_produto,restaurante,positive,8,False
4,conv_00703,"[customer] oi, vc pode me dizer como funciona ...",compra,restaurante,neutral,8,False


,conv_id,text,intent,sector,sentiment,turns,_synthetic
812,syn_100137,[customer] preciso cancelar o curso online que...,cancelamento,,,9,True
1406,syn_100731,"[customer] ola, gostaria de saber sobre aquela...",duvida_produto,,,10,True
289,conv_00664,"[customer] Oi, vc pode me explicar como funcio...",duvida_servico,restaurante,positive,8,False
1606,syn_100931,"[customer] ei, preciso saber uma parada sobre ...",duvida_servico,,,7,True
1937,syn_101262,"[customer] Oi, blz? Preciso cancelar uma consu...",saudacao,,,7,True


Shape: (2120, 7)
Colunas: ['conv_id', 'text', 'intent', 'sector', 'sentiment', 'turns', '_synthetic']

Informações do DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2120 entries, 0 to 2119
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   conv_id     2120 non-null   object
 1   text        2120 non-null   object
 2   intent      2120 non-null   object
 3   sector      2120 non-null   object
 4   sentiment   2120 non-null   object
 5   turns       2120 non-null   int64 
 6   _synthetic  2120 non-null   bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 101.6+ KB

Tipos de dados:


,dtype
conv_id,object
text,object
intent,object
sector,object
sentiment,object
turns,int64
_synthetic,bool



Resumo descritivo:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
conv_id,2120,2120,conv_00384,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
text,2120,2120,"[customer] Oi, vc pode me ajudar? Quero contra...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intent,2120,8,compra,265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sector,2120,9,,1444,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sentiment,2120,4,,1444,NaN,NaN,NaN,NaN,NaN,NaN,NaN
turns,2120.0,NaN,NaN,NaN,7.842925,0.962902,5.0,7.0,8.0,8.0,13.0
_synthetic,2120,2,True,1444,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Valores ausentes, duplicatas e qualidade básica

Além de valores nulos, a checagem inclui strings vazias, textos apenas com espaços, espaços externos, duplicatas exatas, IDs repetidos e cardinalidade. Textos curtos, gírias e erros gramaticais não são tratados como inválidos.

In [4]:
string_columns = df.select_dtypes(include=['object', 'string']).columns.tolist()
missing_summary = pd.DataFrame({
    'valores_ausentes': df.isna().sum(),
    'percentual_ausente': (df.isna().mean() * 100).round(2),
})
display(missing_summary)

string_quality = pd.DataFrame({
    'strings_vazias': {column: df[column].astype('string').eq('').sum() for column in string_columns},
    'apenas_espacos': {column: df[column].astype('string').str.fullmatch(r'\s+').sum() for column in string_columns},
    'espacos_externos': {column: (df[column].astype('string') != df[column].astype('string').str.strip()).sum() for column in string_columns},
})
display(string_quality)

print(f'Duplicatas exatas: {df.duplicated().sum():,}')
print(f'conv_id duplicados: {df["conv_id"].duplicated().sum():,}')
display(df.nunique(dropna=False).rename('valores_unicos').to_frame())

,valores_ausentes,percentual_ausente
conv_id,0,0.0
text,0,0.0
intent,0,0.0
sector,0,0.0
sentiment,0,0.0
turns,0,0.0
_synthetic,0,0.0


,strings_vazias,apenas_espacos,espacos_externos
conv_id,0,0,0
text,0,0,0
intent,0,0,0
sector,1444,0,0
sentiment,1444,0,0


Duplicatas exatas: 0
conv_id duplicados: 0


,valores_unicos
conv_id,2120
text,2120
intent,8
sector,9
sentiment,4
turns,9
_synthetic,2


## 5. Distribuições, textos e turnos

As tabelas mostram as classes usadas pelo dataset sem produzir gráficos, que ficarão para a EDA. `text` é o conteúdo textual, `intent` a intenção, `sector` o setor, `sentiment` o sentimento, `turns` o número de turnos e `_synthetic` a origem do registro.

In [5]:
def distribution_table(column: str) -> pd.DataFrame:
    result = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name='quantidade')
    result['percentual'] = (result['quantidade'] / len(df) * 100).round(2)
    return result

for column in ['intent', 'sector', 'sentiment', '_synthetic']:
    print(f'\nDistribuição de {column}:')
    display(distribution_table(column))

print('Estatísticas de turns:')
display(df['turns'].describe().rename('turns').to_frame())
print('Exemplos de text:')
display(df[['conv_id', 'text', 'intent']].sample(n=5, random_state=42))

text_size = pd.DataFrame({
    'caracteres': df['text'].str.len(),
    'palavras': df['text'].str.split().str.len(),
})
print('Tamanho dos textos:')
display(text_size.describe())


Distribuição de intent:


,intent,quantidade,percentual
0,compra,265,12.5
1,reclamacao,265,12.5
2,cancelamento,265,12.5
3,duvida_produto,265,12.5
4,saudacao,265,12.5
5,elogio,265,12.5
6,suporte_tecnico,265,12.5
7,duvida_servico,265,12.5



Distribuição de sector:


,sector,quantidade,percentual
0,,1444,68.11
1,ecommerce,101,4.76
2,telecom,96,4.53
3,saude,96,4.53
4,financeiro,89,4.20
5,imobiliario,78,3.68
6,educacao,77,3.63
7,restaurante,74,3.49
8,tecnologia,65,3.07



Distribuição de sentiment:


...

,turns
count,2120.000000
mean,7.842925
std,0.962902
min,5.000000
25%,7.000000
50%,8.000000
75%,8.000000
max,13.000000


Exemplos de text:


,conv_id,text,intent
812,syn_100137,[customer] preciso cancelar o curso online que...,cancelamento
1406,syn_100731,"[customer] ola, gostaria de saber sobre aquela...",duvida_produto
289,conv_00664,"[customer] Oi, vc pode me explicar como funcio...",duvida_servico
1606,syn_100931,"[customer] ei, preciso saber uma parada sobre ...",duvida_servico
1937,syn_101262,"[customer] Oi, blz? Preciso cancelar uma consu...",saudacao


Tamanho dos textos:


,caracteres,palavras
count,2120.000000,2120.000000
mean,848.339151,145.919811
std,233.745962,43.605423
min,403.000000,66.000000
25%,674.000000,113.000000
50%,794.500000,136.000000
75%,974.250000,171.000000
max,1866.000000,329.000000


## 6. Comparação entre dados originais e sintéticos

Dados sintéticos não são removidos por sua origem. A comparação abaixo torna explícita a composição e identifica metadados que possam estar disponíveis apenas para conversas originais.

In [6]:
origin_summary = distribution_table('_synthetic').assign(origem=lambda table: table['_synthetic'].map({False: 'original', True: 'sintético'}))
display(origin_summary[['origem', 'quantidade', 'percentual']])

for column in ['intent', 'sentiment', 'sector']:
    print(f'\nDistribuição de {column} por origem:')
    display(pd.crosstab(df['_synthetic'], df[column], dropna=False))

origin_metrics = (
    df.assign(caracteres=df['text'].str.len(), palavras=df['text'].str.split().str.len())
    .groupby('_synthetic')[['caracteres', 'palavras', 'turns']]
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .round(2)
)
print('Tamanho médio dos textos e turns médio por origem:')
display(origin_metrics)

,origem,quantidade,percentual
0,sintético,1444,68.11
1,original,676,31.89



Distribuição de intent por origem:


intent,cancelamento,compra,duvida_produto,duvida_servico,elogio,reclamacao,saudacao,suporte_tecnico
_synthetic,,,,,,,,
False,79,85,80,87,85,84,88,88
True,186,180,185,178,180,181,177,177



Distribuição de sentiment por origem:


sentiment,,negative,neutral,positive
_synthetic,,,,
False,0,223,231,222
True,1444,0,0,0



Distribuição de sector por origem:


sector,,ecommerce,educacao,financeiro,imobiliario,restaurante,saude,tecnologia,telecom
_synthetic,,,,,,,,,
False,0,101,77,89,78,74,96,65,96
True,1444,0,0,0,0,0,0,0,0


Tamanho médio dos textos e turns médio por origem:


caracteres                             palavras                     \
                count     mean  median  min   max    count    mean median min   
_synthetic                                                                      
False             676  1091.57  1089.0  475  1866      676  192.83  192.0  86   
True             1444   734.47   723.0  403  1258     1444  123.96  122.0  66   

                turns                       
            max count  mean median min max  
_synthetic                                  
False       329   676  7.82    8.0   6   8  
True        214  1444  7.85    8.0   5  13

## Diagnóstico de Qualidade dos Dados

| Verificação | Evidência | Decisão | Justificativa |
|---|---|---|---|
| Estrutura | 2.120 linhas e 7 colunas; `text`, `intent`, `sector`, `sentiment`, `turns` e `_synthetic` estão presentes. | Manter. | Atende aos requisitos de volume, texto e intenção. |
| Valores nulos | 0 em todas as colunas. | Não imputar nem remover. | Não há nulos reais. |
| Strings vazias | `sector` e `sentiment` têm 1.444 strings vazias cada; demais colunas têm 0. | Converter somente essas strings vazias em ausentes no arquivo processado. | Elas ocorrem em todas as conversas sintéticas, onde esses metadados não foram fornecidos; vazio não é uma categoria. |
| Strings apenas com espaços e espaços externos | 0 em todas as colunas textuais. | Não aplicar `strip`. | Não há evidência que justifique alterar strings. |
| Duplicatas | 0 linhas duplicadas e 0 `conv_id` duplicados. | Não remover registros. | Não há duplicatas a tratar. |
| Intenções | 8 valores, 265 conversas por intenção. | Manter os rótulos. | Não há grafias inconsistentes observadas; as classes estão balanceadas. |
| Setor e sentimento | Os 676 originais possuem metadados; os 1.444 sintéticos têm valor vazio. | Preservar todas as conversas e representar os vazios como ausentes. | A lacuna é ligada à origem, não uma razão para excluir dados sintéticos. |
| `_synthetic` | 676 originais (31,89%) e 1.444 sintéticas (68,11%). | Manter ambas as origens. | A documentação do dataset informa que os sintéticos foram adicionados para balancear as intenções. |
| `turns` e tamanho textual | `turns` varia de 5 a 13 (mediana 8); textos variam de 403 a 1.866 caracteres (mediana 794,5). | Manter. | Os valores são compatíveis com conversas multi-turno; não há extremos inválidos evidentes. |

## 7. Limpeza justificada e salvamento

A única limpeza aplicada é substituir a string vazia por valor ausente em `sector` e `sentiment`. Não há remoção de duplicatas, correção linguística, filtragem de setor/intenção ou exclusão de dados sintéticos.

In [7]:
clean_df = df.copy()
for column in ['sector', 'sentiment']:
    clean_df[column] = clean_df[column].replace('', pd.NA)

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f'Arquivo processado salvo em: {PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT)}')
display(clean_df.isna().sum().rename('ausentes_apos_limpeza').to_frame())
assert len(clean_df) == len(df), 'A limpeza não deve remover linhas.'
assert clean_df.duplicated().sum() == 0, 'A limpeza não deve criar duplicatas.'

Arquivo processado salvo em: data/processed/talkex_clean.csv


,ausentes_apos_limpeza
conv_id,0
text,0
intent,0
sector,1444
sentiment,1444
turns,0
_synthetic,0


## Próximos passos

A próxima etapa deve criar um notebook separado de EDA com pelo menos três visualizações e três hipóteses baseadas nestes resultados. Ao interpretar `sector` e `sentiment`, considere que esses metadados não estão disponíveis nas conversas sintéticas.